# 111–115 學測與 115 起適用考試說明：英語例句、Danube 500M 微調與 iPhone 13 網站串接

將本 `.ipynb` 下載後直接上傳 [Google Colab](https://colab.research.google.com/)，選擇 **執行階段 → 變更執行階段類型 → T4 GPU**，從上到下執行。筆記本內嵌 helper、原創示範資料與課綱匯出器；不必先把這次修改推送 GitHub。

本流程固定 `jasoart/E` 的 `restore-old-site` 已檢查版本與 Hugging Face 模型的實際 commit。預設標準 LoRA，選用 QLoRA/NF4 或 PiSSA/LoftQ；部署另行合併 fp32，再量化為 **ONNX MatMulNBits q4**，供 Transformers.js 3.8.1 的 WASM 使用。訓練 NF4 與部署 q4 是不同格式。

模型輸出一個 8–24 字的英文例句；繁體中文詞義提供語境。Danube 官方模板不支援 system role，因此指示合併在 user turn；網站與訓練使用同一份 prompt。本輪研究參考 111–115 年試卷及 115 學年度起適用考試說明；**內嵌訓練資料仍是既有 78 句以 115 為種子的原創示範，未擴充成五年訓練集**，不能證明微調有效或取得學測程度。可一次上傳多份 PDF 作八詞重複比對；考試說明內的歷年示例也納入防照抄參考，不能當成 111–115 各年實際考題。PDF 與抽取文字不會成為微調語料或候選模型 ZIP。

官方考試說明以常用約 4,500 字、第 1–5 級為核心，偶爾包含第 6 級以上；180–400 字是題組選文長度。**8–24 詞與程式的詞表比例門檻都是單句練習設定，不是官方难度認證**。五年試卷也要求語域、立場、時態視點、跨句指涉和圖文理解；這個單句流程未驗證這些完整能力。

筆記本只產生本機 ZIP，**不會上傳模型、試卷、學習紀錄或 Token**。500M 模型仍可能有文法、詞義和例句難度問題；自動指標是篩選工具，最終需英文教師與 iPhone 13 Safari 實測。


In [ ]:
# 每次建立獨立目錄，不覆寫原有資料；Colab VM 不使用 Git worktree。
import gc, json, os, shutil, subprocess, sys, uuid
from pathlib import Path
from google.colab import files
RUN_DIR = Path('/content/gsat-runs') / uuid.uuid4().hex[:12]
RUN_DIR.mkdir(parents=True)
SOURCE_DIR = RUN_DIR / 'source'
BASE_REPO_SHA = '7513d5ad546b79a2f21c8b0e604206efb355c3be'
METHOD = 'lora'               # 'lora' 或 'qlora'（T4 使用 NF4 訓練）
INITIALIZATION = 'standard'  # 'standard'；lora可選'pissa'，qlora可選'loftq'
DEMO_MODE = True             # 示範20steps；自備充分資料才改False
ADD_GENERIC_DEMO = False    # 預設仍用既有78句115種子原創示範，非五年訓練集
UPLOAD_MY_JSONL = False      # True時上傳自己有權使用的JSONL；不收集網站使用者資料
UPLOAD_REFERENCE_PDF = False # True時一次上傳1+份PDF（111–115試卷／115起適用說明），僅在本VM比對
MODEL_ID = 'local/gsat-danube3-500m-example-q4' # 之後人工發布時改成自己的公開repo ID
HUMAN_REVIEW = None          # 例如 {'reviewed_examples':50,'grammar_accuracy':0.96}
IPHONE_REVIEW = None         # 完成實測才填 {'device':'iPhone 13','passed':True}
print('本機執行目錄:', RUN_DIR)


## 1. 安裝固定版本並檢查 GPU

套件版本固定，保留 Colab 已安裝且相容的 PyTorch/CUDA，避免更換顯示卡驅動。固定版本不是「最新版」聲明。執行後若 Colab 提示重新啟動，重新啟動一次並從上方設定格開始執行。`pip check` 會列出預裝環境的相依性問題，若牽涉本流程套件應先排除。


In [ ]:
EMBEDDED_FILES = {'gsat_workflow.py': '"""GSAT example-sentence training/export helpers; heavyweight imports stay optional.\n\nThis module never publishes a model or collects website learning records. The pure\ndata, prompt, evaluation, and release-gate functions can be tested without a GPU.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nimport shutil\nfrom pathlib import Path\n\nBASE_MODEL = "h2oai/h2o-danube3-500m-chat"\nBASE_REVISION = "c202f976c26875541e738ea978c8158fa536da9a"\nMAX_INPUT_TOKENS = 384\nMAX_NEW_TOKENS = 96\nSEED = 115\nSYSTEM_PROMPT = (\n    "You write accurate English example sentences for Taiwanese GSAT learners. "\n    "Return only one sentence, with no explanation, translation, or list."\n)\nPACKAGES = (\n    "transformers==4.55.4", "peft==0.17.1", "accelerate==1.10.1",\n    "bitsandbytes==0.46.1", "optimum==2.1.0", "optimum-onnx==0.1.0",\n    "onnx==1.17.0", "onnxruntime==1.21.0", "pypdf==5.9.0",\n)\nWORD_RE = re.compile(r"[A-Za-z]+(?:[\'’\\-][A-Za-z]+)*")\nGRAMMAR_INSTRUCTIONS = {\n    "auto": "Choose a suitable simple or complex sentence structure.",\n    "contextual_collocation": "Use a natural collocation in a familiar context.",\n    "adverbial_cause_time": "Use a cause or time adverbial clause (because, since, when, or after).",\n    "concession_contrast": "Use a concession or contrast connector (although, while, or however).",\n    "relative_clause": "Use a relative clause (who, which, or that).",\n    "passive_voice": "Use a natural passive construction.",\n    "participial_modifier": "Use a natural participial phrase.",\n    "noun_clause_reporting": "Use a reporting verb followed by a noun clause (that or whether).",\n    "purpose_result": "Use a purpose or result construction (so that, in order to, or so...that).",\n    "comparison_degree": "Use a natural comparison or degree construction.",\n    "conditional_modal": "Use a conditional clause or a modal verb.",\n    "discourse_linking": "Use a suitable discourse connector to link ideas within one sentence.",\n    "perfect_tense": "Use a natural present or past perfect construction.",\n}\n\n\ndef read_jsonl(path):\n    records = []\n    for number, line in enumerate(Path(path).read_text(encoding="utf-8").splitlines(), 1):\n        if not line.strip():\n            continue\n        try:\n            record = json.loads(line)\n        except json.JSONDecodeError as exc:\n            raise ValueError(f"Invalid JSON at line {number}") from exc\n        validate_record(record)\n        records.append(record)\n    if not records:\n        raise ValueError("The dataset is empty")\n    ids = [r["id"] for r in records]\n    if len(set(ids)) != len(ids):\n        raise ValueError("Dataset IDs must be unique")\n    return records\n\n\ndef validate_record(record):\n    if not isinstance(record, dict):\n        raise ValueError("Each record must be an object")\n    for key in ("id", "target_word", "sentence", "source", "license"):\n        if not isinstance(record.get(key), str) or not record[key].strip():\n            raise ValueError(f"Missing or invalid {key}")\n    if len(record["sentence"]) > 600 or not WORD_RE.fullmatch(record["target_word"]):\n        raise ValueError("Invalid target word or oversized answer")\n    if not isinstance(record.get("level"), int) or isinstance(record["level"], bool) or not 1 <= record["level"] <= 6:\n        raise ValueError("level must be an integer from 1 to 6")\n    forms = record.get("target_forms", [record["target_word"]])\n    if not isinstance(forms, list) or not forms or not all(isinstance(f, str) and WORD_RE.fullmatch(f) for f in forms):\n        raise ValueError("target_forms must contain valid English word forms")\n    if record.get("grammar_tag", "auto") not in GRAMMAR_INSTRUCTIONS:\n        raise ValueError("Unsupported grammar_tag")\n    if not contains_target(record["sentence"], forms):\n        raise ValueError(f"Answer does not contain the target: {record[\'id\']}")\n    if not 8 <= len(english_words(record["sentence"])) <= 24:\n        raise ValueError(f"Answer must contain 8 to 24 words: {record[\'id\']}")\n    if re.search(r"https?://|\\b[\\w.+-]+@[\\w.-]+\\.[A-Za-z]{2,}\\b", record["sentence"]):\n        raise ValueError("Training examples must not contain URLs or email addresses")\n\n\ndef english_words(text):\n    return [w.lower().replace("’", "\'") for w in WORD_RE.findall(text)]\n\n\ndef contains_target(text, forms):\n    words = set(english_words(text))\n    return any(form.lower().replace("’", "\'") in words for form in forms)\n\n\ndef split_records(records, seed=SEED):\n    """Hash the target, never the row: every sense/form stays in one partition."""\n    partitions = {"train": [], "validation": [], "test": []}\n    for record in records:\n        key = record["target_word"].strip().casefold()\n        value = int(hashlib.sha256(f"{seed}:{key}".encode()).hexdigest()[:8], 16) / 2**32\n        partition = "train" if value < 0.8 else "validation" if value < 0.9 else "test"\n        partitions[partition].append(record)\n    for name in partitions:\n        partitions[name].sort(key=lambda row: row["id"])\n    word_sets = [{r["target_word"].casefold() for r in rows} for rows in partitions.values()]\n    if any(a & b for i, a in enumerate(word_sets) for b in word_sets[i + 1:]):\n        raise AssertionError("Target-word leakage")\n    return partitions\n\n\ndef curriculum_index(curriculum):\n    rows = curriculum.get("vocabulary", curriculum.get("words", [])) if isinstance(curriculum, dict) else curriculum\n    if not isinstance(rows, list) or not rows:\n        raise ValueError("Curriculum must contain a nonempty list of word records")\n    index = {row["word"].casefold(): row for row in rows}\n    for row in rows:\n        for alternative in row["word"].casefold().split("/"):\n            primary = re.sub(r"\\([^)]*\\)", "", alternative).strip()\n            if primary:\n                index.setdefault(primary, row)\n            # Repository notation improve(ment) represents improve/improvement.\n            if re.fullmatch(r"[a-z]+\\([a-z]+\\)", alternative):\n                index.setdefault(alternative.replace("(", "").replace(")", ""), row)\n    return index\n\n\ndef clean(value, maximum):\n    return re.sub(r"\\s+", " ", re.sub(r"[\\x00-\\x1f\\x7f]", " ", str(value or ""))).strip()[:maximum]\n\n\ndef build_messages(record, curriculum=None):\n    meta = (curriculum or {}).get(record["target_word"].casefold(), {})\n    # The browser uses the same limits and exact prompt strings.\n    word = clean(re.sub(r"\\([^)]*\\)", "", record["target_word"].split("/")[0]), 80)\n    pos = clean(record.get("partOfSpeech", meta.get("partOfSpeech", "")), 40)\n    meaning = clean(record.get("meaning_zh", meta.get("meaning", "")), 120)\n    level = min(6, max(5, int(record.get("level", meta.get("level", 5)))))\n    patterns = record.get("patterns", meta.get("patterns", []))\n    patterns = [clean(x, 100) for x in patterns[:2] if clean(x, 100)]\n    topic = clean(record.get("topic"), 100) or "a familiar school, daily life, or social situation"\n    tag = record.get("grammar_tag", "auto")\n    grammar = GRAMMAR_INSTRUCTIONS.get(tag, GRAMMAR_INSTRUCTIONS["auto"])\n    user = (\n        f\'Write one natural English sentence of 8 to 24 words using "{word}".\\n\'\n        f"Part of speech: {pos}. Meaning: {meaning}.\\n"\n        f"Keep the vocabulary within CEEC levels 1 to {level}.\\n"\n        f"Useful collocations: {\'; \'.join(patterns) or \'(none supplied)\'}.\\n"\n        f"Topic: {topic}.\\n"\n        f"Grammar: {grammar}\\n"\n        "Use the target in a complete sentence. Return the English sentence only."\n    )\n    # Danube\'s official template supports alternating user/assistant roles and\n    # explicitly rejects system messages. Fold instructions into the user turn.\n    return [{"role": "user", "content": SYSTEM_PROMPT + "\\n\\n" + user}]\n\n\ndef tokenize_answer(tokenizer, messages, answer):\n    """Keep assistant-only loss and reject incomplete/truncated training answers."""\n    prefix = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)\n    answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]\n    if tokenizer.eos_token_id is None:\n        raise ValueError("Tokenizer requires an EOS token")\n    answer_ids = answer_ids + [tokenizer.eos_token_id]\n    if len(prefix) > MAX_INPUT_TOKENS:\n        raise ValueError("Prompt exceeds the website token budget; shorten metadata")\n    if len(answer_ids) > MAX_NEW_TOKENS:\n        raise ValueError("Answer exceeds the website output budget; shorten the answer")\n    return {\n        "input_ids": prefix + answer_ids,\n        "attention_mask": [1] * (len(prefix) + len(answer_ids)),\n        "labels": [-100] * len(prefix) + answer_ids,\n    }\n\n\nclass AnswerOnlyCollator:\n    def __init__(self, pad_token_id):\n        self.pad_token_id = pad_token_id\n\n    def __call__(self, rows):\n        import torch\n        width = max(len(row["input_ids"]) for row in rows)\n        result = {"input_ids": [], "attention_mask": [], "labels": []}\n        for row in rows:\n            padding = width - len(row["input_ids"])\n            result["input_ids"].append(row["input_ids"] + [self.pad_token_id] * padding)\n            result["attention_mask"].append(row["attention_mask"] + [0] * padding)\n            result["labels"].append(row["labels"] + [-100] * padding)\n        return {key: torch.tensor(values, dtype=torch.long) for key, values in result.items()}\n\n\ndef ngrams(text, n=8):\n    words = english_words(text)\n    return {tuple(words[index:index + n]) for index in range(max(0, len(words) - n + 1))}\n\n\ndef reference_ngram_index(pdf_paths):\n    """Index local no-copy references, without exporting their text or 8-grams.\n\n    Each distinct PDF must yield at least one English 8-gram. Pages are indexed\n    separately so a fragment cannot cross a page boundary. Only ``metadata`` is\n    JSON-serializable and suitable for the run lock; ``ngrams`` stays in memory.\n    Content-identical uploads are extracted and counted once, while their names\n    remain visible for auditing. An exam specification can contain illustrative\n    past questions: it is a reference, not evidence of that year\'s actual exam.\n    """\n    if isinstance(pdf_paths, (str, Path)):\n        raise ValueError("Pass a nonempty list of reference PDF paths")\n    paths = [Path(path) for path in pdf_paths]\n    if not paths:\n        raise ValueError("At least one reference PDF is required")\n    for path in paths:\n        if path.suffix.lower() != ".pdf":\n            raise ValueError(f"Reference must be a PDF: {path.name}")\n        if not path.is_file():\n            raise ValueError(f"Reference PDF is not a file: {path.name}")\n        if path.stat().st_size == 0:\n            raise ValueError(f"Reference PDF is empty: {path.name}")\n    from pypdf import PdfReader\n\n    result, documents, by_hash = set(), [], {}\n    for path in paths:\n        digest = sha256_file(path)\n        if digest in by_hash:\n            by_hash[digest]["duplicate_filenames"].append(path.name)\n            continue\n        try:\n            pages = PdfReader(str(path)).pages\n            page_count = len(pages)\n        except Exception as exc:\n            raise ValueError(f"Cannot read reference PDF: {path.name}") from exc\n        if page_count == 0:\n            raise ValueError(f"Reference PDF has no pages: {path.name}")\n        document_ngrams, pages_with_8grams = set(), 0\n        for number, page in enumerate(pages, 1):\n            try:\n                page_ngrams = ngrams(page.extract_text() or "")\n            except Exception as exc:\n                raise ValueError(f"Cannot extract reference PDF {path.name}, page {number}") from exc\n            document_ngrams.update(page_ngrams)\n            pages_with_8grams += bool(page_ngrams)\n        if not document_ngrams:\n            raise ValueError(f"No English 8-word reference text extracted from {path.name}; check PDF manually")\n        document = {\n            "filename": path.name, "sha256": digest, "page_count": page_count,\n            "pages_with_8grams": pages_with_8grams,\n            "english_8gram_count": len(document_ngrams), "duplicate_filenames": [],\n        }\n        documents.append(document)\n        by_hash[digest] = document\n        result.update(document_ngrams)\n    return {"ngrams": result, "metadata": {\n        "role": "no-copy-overlap-reference-only", "ngram_words": 8,\n        "uploaded_document_count": len(paths), "unique_document_count": len(documents),\n        "duplicate_document_count": len(paths) - len(documents),\n        "page_count": sum(document["page_count"] for document in documents),\n        "english_8gram_count": len(result), "documents": documents,\n    }}\n\n\ndef reference_ngrams(pdf_path):\n    """Backward-compatible single-PDF wrapper; never creates SFT rows."""\n    return reference_ngram_index([pdf_path])["ngrams"]\n\n\ndef build_coverage_index(curriculum, lexical_levels=None):\n    """CEEC lexical proxy; the small morphology list cannot certify CEFR level."""\n    index = dict(lexical_levels or {})\n    for word, row in curriculum.items():\n        level = int(row.get("level", 6))\n        forms = row.get("target_forms") or [word]\n        # Conservative, explicitly reported morphology heuristic for regular forms.\n        if re.fullmatch(r"[a-z]+", word) and len(word) > 2:\n            forms = [*forms, word + "s", word + "ed", word + "ing"]\n            if word.endswith("e"):\n                forms += [word + "d", word[:-1] + "ing"]\n            if word.endswith("y") and word[-2] not in "aeiou":\n                forms += [word[:-1] + "ies", word[:-1] + "ied"]\n        for form in forms:\n            for token in english_words(form):\n                index[token] = min(level, index.get(token, 6))\n    return index\n\n\ndef score_sentence(record, sentence, coverage, reference=None):\n    words = english_words(sentence)\n    ceiling = min(6, max(5, int(record["level"])))\n    # Abbreviations and names may be flagged; a human must review flagged outputs.\n    unknown = [word for word in words if coverage.get(word, 99) > ceiling]\n    terminators = re.findall(r"[.!?](?=\\s|$)", sentence.strip())\n    one_sentence = len(terminators) == 1 and not re.search(r"\\n\\s*[-*\\d]|[\\u3400-\\u9fff]", sentence)\n    overlap = bool(ngrams(sentence) & reference) if reference is not None else None\n    return {\n        "target_match": contains_target(sentence, record.get("target_forms", [record["target_word"]])),\n        "length_ok": 8 <= len(words) <= 24,\n        "single_sentence": one_sentence,\n        "word_count": len(words),\n        "ceec_coverage": (len(words) - len(unknown)) / len(words) if words else 0,\n        "unknown_or_overlevel_words": unknown,\n        "copied_8gram": overlap,\n    }\n\n\ndef summarize_scores(rows):\n    if not rows:\n        raise ValueError("Cannot evaluate zero held-out prompts")\n    report = {"count": len(rows)}\n    for key in ("target_match", "length_ok", "single_sentence", "ceec_coverage"):\n        report[key] = sum(float(row["score"][key]) for row in rows) / len(rows)\n    checked = [row for row in rows if row["score"]["copied_8gram"] is not None]\n    report["reference_overlap_checked"] = len(checked) == len(rows)\n    report["copied_8gram_rate"] = sum(row["score"]["copied_8gram"] for row in checked) / len(checked) if checked else None\n    return report\n\n\ndef release_gate(partitions, reports, artifact_bytes=None, human_review=None, iphone_review=None):\n    """Fail closed: automated proxies do not establish grammatical/device quality."""\n    reasons = []\n    train_words = len({row["target_word"].casefold() for row in partitions["train"]})\n    test_words = len({row["target_word"].casefold() for row in partitions["test"]})\n    if train_words < 200 or test_words < 50:\n        reasons.append("Demo data: require >=200 training words and >=50 held-out test words")\n    if not all(stage in reports for stage in ("baseline", "finetuned", "q4")):\n        reasons.append("Baseline, finetuned, and q4 evaluations must all complete")\n    else:\n        fine, quant = reports["finetuned"], reports["q4"]\n        if any(reports[stage].get("count") != len(partitions["test"]) for stage in ("baseline", "finetuned", "q4")):\n            reasons.append("Each stage must evaluate every held-out test prompt")\n        prompt_hashes = {reports[stage].get("prompt_sha256") for stage in ("baseline", "finetuned", "q4")}\n        if None in prompt_hashes or len(prompt_hashes) != 1:\n            reasons.append("All evaluation stages must use the exact same held-out prompts")\n        for key, minimum in (("target_match", 0.9), ("length_ok", 0.9), ("single_sentence", 0.95), ("ceec_coverage", 0.95)):\n            if quant[key] < minimum:\n                reasons.append(f"q4 {key} below {minimum}")\n            if quant[key] < fine[key] - 0.02:\n                reasons.append(f"q4 {key} regressed by more than 2 percentage points")\n        if not quant.get("reference_overlap_checked") or quant.get("copied_8gram_rate") != 0:\n            reasons.append("Reference overlap check missing or copied eight-word phrases found")\n    if artifact_bytes is None or artifact_bytes > 768 * 1024**2:\n        reasons.append("Model artifact size is unknown or exceeds 768 MiB")\n    if not human_review or human_review.get("reviewed_examples", 0) < 50 or human_review.get("grammar_accuracy", 0) < 0.95:\n        reasons.append("Require human review of >=50 examples with >=95% grammar/sense accuracy")\n    if not iphone_review or iphone_review.get("device") != "iPhone 13" or not iphone_review.get("passed"):\n        reasons.append("iPhone 13 Safari memory, offline, cancellation and latency checks pending")\n    return {"release_eligible": not reasons, "reasons": reasons, "train_words": train_words, "test_words": test_words}\n\n\ndef write_json(path, data):\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef lock_model_revision(output_dir, dataset_paths):\n    import importlib.metadata\n    from huggingface_hub import HfApi\n    from transformers import AutoConfig\n    info = HfApi().model_info(BASE_MODEL, revision=BASE_REVISION)\n    if not re.fullmatch(r"[0-9a-f]{40}", info.sha or ""):\n        raise ValueError("Unable to resolve an immutable model revision")\n    if info.sha != BASE_REVISION:\n        raise ValueError("Resolved model revision differs from the reviewed preset")\n    config = AutoConfig.from_pretrained(BASE_MODEL, revision=info.sha, trust_remote_code=False)\n    if config.model_type != "llama" or "LlamaForCausalLM" not in (config.architectures or []):\n        raise ValueError(f"Export pipeline expects LlamaForCausalLM, found {config.architectures}")\n    license_id = (info.card_data or {}).get("license")\n    if license_id != "apache-2.0":\n        raise ValueError(f"Review upstream license before continuing: {license_id}")\n    lock = {\n        "base_model": BASE_MODEL, "base_revision": info.sha, "base_license": license_id,\n        "model_type": config.model_type, "max_position_embeddings": config.max_position_embeddings,\n        "datasets": [{"name": Path(path).name, "sha256": sha256_file(path)} for path in dataset_paths],\n        "seed": SEED, "packages": {},\n    }\n    for package in ("torch", "transformers", "peft", "accelerate", "bitsandbytes", "optimum", "optimum-onnx", "onnx", "onnxruntime"):\n        lock["packages"][package] = importlib.metadata.version(package)\n    write_json(Path(output_dir) / "run_lock.json", lock)\n    return lock\n\n\ndef gpu_guard():\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError("Select Colab Runtime > Change runtime type > T4 GPU before training")\n    properties = torch.cuda.get_device_properties(0)\n    if properties.total_memory < 8 * 1024**3:\n        raise RuntimeError("At least 8 GiB GPU VRAM is required by this preset")\n    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n    return {"device": properties.name, "vram_gib": round(properties.total_memory / 1024**3, 1), "dtype": dtype}\n\n\ndef load_baseline_model(lock):\n    from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed\n    gpu = gpu_guard()\n    set_seed(SEED)\n    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=lock["base_revision"], trust_remote_code=False)\n    if not tokenizer.chat_template:\n        raise RuntimeError("Upstream chat template missing")\n    tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = "right"\n    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, revision=lock["base_revision"],\n        torch_dtype=gpu["dtype"], device_map={"": 0}, trust_remote_code=False, attn_implementation="eager")\n    return model, tokenizer\n\n\ndef load_training_model(lock, method="lora", initialization="standard", output_dir=None):\n    import torch\n    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, replace_lora_weights_loftq\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed\n    if method not in ("lora", "qlora") or initialization not in ("standard", "pissa", "loftq"):\n        raise ValueError("Unsupported training method/initialization")\n    if initialization == "pissa" and method != "lora":\n        raise ValueError("This preset initializes PiSSA on nonquantized weights; use lora")\n    if initialization == "loftq" and method != "qlora":\n        raise ValueError("LoftQ weight replacement requires the qlora preset")\n    gpu = gpu_guard()\n    set_seed(SEED)\n    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=lock["base_revision"], trust_remote_code=False)\n    if not tokenizer.chat_template:\n        raise RuntimeError("Upstream chat template missing; do not invent a replacement")\n    tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = "right"\n    kwargs = dict(revision=lock["base_revision"], torch_dtype=gpu["dtype"], trust_remote_code=False,\n                  attn_implementation="eager", device_map={"": 0})\n    if method == "qlora":\n        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=gpu["dtype"])\n    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)\n    if method == "qlora":\n        model = prepare_model_for_kbit_training(model, gradient_checkpointing_kwargs={"use_reentrant": False})\n    model.config.use_cache = False\n    # PEFT0.17.1 LoftQ replacement assigns residual factors without rescaling;\n    # alpha/r must be one, otherwise the initialized residual is multiplied.\n    config = LoraConfig(task_type="CAUSAL_LM", r=16, lora_alpha=16 if initialization == "loftq" else 32, target_modules="all-linear",\n        lora_dropout=0 if initialization == "pissa" else 0.05, bias="none",\n        init_lora_weights="pissa_niter_4" if initialization == "pissa" else True)\n    model = get_peft_model(model, config)\n    if initialization == "pissa":\n        # PiSSA changes the frozen residual; saving this initial adapter enables\n        # conversion to a standard adapter for reload against the ORIGINAL base.\n        if output_dir is None:\n            raise ValueError("PiSSA requires an output directory to preserve the initial adapter")\n        model.save_pretrained(str(Path(output_dir) / "pissa_initial"))\n    if initialization == "loftq":\n        from huggingface_hub import snapshot_download\n        # PEFT\'s implicit loader looks up refs/main. A fresh Colab only has the\n        # locked SHA snapshot; pass it explicitly to avoid drifting or failing.\n        snapshot = snapshot_download(BASE_MODEL, revision=lock["base_revision"], local_files_only=True)\n        replace_lora_weights_loftq(model, model_path=str(snapshot))\n    model.print_trainable_parameters()\n    return model, tokenizer, gpu\n\n\ndef evaluate_model(model, tokenizer, records, curriculum, coverage, output_path, reference=None, device=None):\n    import torch\n    rows = []\n    if hasattr(model, "eval"):\n        model.eval()\n    with torch.inference_mode():\n        for record in records:\n            messages = build_messages(record, curriculum)\n            token_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)\n            if len(token_ids) > MAX_INPUT_TOKENS:\n                raise ValueError(f"Evaluation prompt too long: {record[\'id\']}")\n            inputs = {"input_ids": torch.tensor([token_ids], dtype=torch.long), "attention_mask": torch.ones((1, len(token_ids)), dtype=torch.long)}\n            if device:\n                inputs = {key: value.to(device) for key, value in inputs.items()}\n            generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,\n                repetition_penalty=1.1, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id, use_cache=True)\n            sentence = tokenizer.decode(generated[0, len(token_ids):], skip_special_tokens=True).strip()\n            rows.append({"id": record["id"], "target_word": record["target_word"], "messages": messages,\n                         "sentence": sentence, "score": score_sentence(record, sentence, coverage, reference)})\n    summary = summarize_scores(rows)\n    summary["prompt_sha256"] = hashlib.sha256(json.dumps(\n        [{"id": row["id"], "messages": row["messages"]} for row in rows],\n        ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()\n    write_json(output_path, {"summary": summary, "examples": rows,\n               "limitations": "CEEC morphology coverage, structure and overlap are proxies; grammar, sense and exam similarity require human review."})\n    return summary\n\n\ndef train_adapter(model, tokenizer, partitions, curriculum, output_dir, gpu, demo=True, initialization="standard"):\n    from transformers import Trainer, TrainingArguments\n    encoded = {}\n    for name in ("train", "validation"):\n        if not partitions[name]:\n            raise ValueError(f"No {name} rows; add more distinct target words")\n        encoded[name] = [tokenize_answer(tokenizer, build_messages(row, curriculum), row["sentence"]) for row in partitions[name]]\n    # Tiny bundled data demonstrates the workflow, never claims measurable gains.\n    arguments = TrainingArguments(output_dir=str(Path(output_dir) / "checkpoints"),\n        per_device_train_batch_size=2, per_device_eval_batch_size=2, gradient_accumulation_steps=4,\n        learning_rate=2e-4, max_steps=20 if demo else -1, num_train_epochs=3,\n        warmup_ratio=0.05, weight_decay=0.01, lr_scheduler_type="cosine", optim="adamw_torch",\n        fp16=str(gpu["dtype"]) == "torch.float16", bf16=str(gpu["dtype"]) == "torch.bfloat16",\n        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},\n        eval_strategy="steps" if demo else "epoch", eval_steps=10 if demo else None,\n        save_strategy="no", logging_steps=5, report_to="none", seed=SEED, data_seed=SEED,\n        dataloader_num_workers=0, remove_unused_columns=False)\n    trainer = Trainer(model=model, args=arguments, train_dataset=encoded["train"], eval_dataset=encoded["validation"],\n                      data_collator=AnswerOnlyCollator(tokenizer.pad_token_id))\n    trainer.train()\n    adapter_path = Path(output_dir) / "adapter"\n    save_kwargs = {}\n    if initialization == "pissa":\n        save_kwargs["path_initial_model_for_weight_conversion"] = str(Path(output_dir) / "pissa_initial")\n    model.save_pretrained(adapter_path, safe_serialization=True, **save_kwargs)\n    tokenizer.save_pretrained(adapter_path)\n    return adapter_path\n\n\ndef merge_adapter(lock, adapter_path, merged_dir):\n    import torch\n    from peft import PeftModel\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    # Reload the pinned ORIGINAL base in fp32. Do not merge into bitsandbytes NF4.\n    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, revision=lock["base_revision"],\n        torch_dtype=torch.float32, device_map={"": "cpu"}, trust_remote_code=False, attn_implementation="eager")\n    merged = PeftModel.from_pretrained(base, str(adapter_path)).merge_and_unload(safe_merge=True)\n    merged.config.use_cache = True\n    merged.save_pretrained(merged_dir, safe_serialization=True)\n    AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=False).save_pretrained(merged_dir)\n    return merged\n\n\ndef export_q4(merged_dir, artifact_dir):\n    import onnx\n    import onnxruntime as ort\n    from optimum.exporters.onnx import main_export\n    from onnxruntime.quantization.matmul_4bits_quantizer import MatMul4BitsQuantizer\n    artifact_dir = Path(artifact_dir)\n    if artifact_dir.exists() and any(artifact_dir.iterdir()):\n        raise ValueError("Choose an empty artifact directory; preserve existing user artifacts")\n    fp32_dir = artifact_dir.parent / "onnx_fp32"\n    if fp32_dir.exists() and any(fp32_dir.iterdir()):\n        raise ValueError("Choose a new run directory; fp32 export already exists")\n    # --no-post-process equivalent is necessary for Transformers.js one-session\n    # decoding: a single model.onnx with optional zero-length past inputs.\n    main_export(model_name_or_path=str(merged_dir), output=str(fp32_dir), task="text-generation-with-past",\n                opset=21, device="cpu", no_post_process=True, do_validation=True,\n                model_kwargs={"use_cache": True})\n    graphs = list(fp32_dir.glob("*.onnx"))\n    if len(graphs) != 1 or graphs[0].name != "model.onnx":\n        raise RuntimeError(f"Unexpected exporter output {[p.name for p in graphs]}; expected one model.onnx")\n    graph = onnx.load(str(graphs[0]), load_external_data=True)\n    names = {value.name for value in graph.graph.input}\n    if not {"input_ids", "attention_mask", "position_ids"}.issubset(names) or not any(n.startswith("past_key_values.") for n in names):\n        raise RuntimeError("ONNX graph is missing Transformers.js causal decoder inputs")\n    if not any(value.name.startswith("present.") for value in graph.graph.output):\n        raise RuntimeError("ONNX graph is missing cache outputs")\n    quantizer = MatMul4BitsQuantizer(graph, block_size=32, is_symmetric=True, accuracy_level=0,\n                                   op_types_to_quantize=("MatMul",))\n    quantizer.process()\n    q4 = quantizer.model.model\n    if not any(node.op_type == "MatMulNBits" and node.domain == "com.microsoft" for node in q4.graph.node):\n        raise RuntimeError("No 4-bit MatMulNBits nodes were produced")\n    artifact_dir.mkdir(parents=True, exist_ok=True)\n    onnx_dir = artifact_dir / "onnx"\n    onnx_dir.mkdir()\n    model_path = onnx_dir / "model_q4.onnx"\n    # Embedding/cache remain fp32. Keep one embedded graph below the 2 GiB limit;\n    # this avoids external-data naming differences between browser/Python tools.\n    if q4.ByteSize() >= 2 * 1024**3:\n        raise RuntimeError("q4 exceeds 2 GiB: this conservative browser preset requires embedded tensors")\n    onnx.save_model(q4, str(model_path), save_as_external_data=False)\n    onnx.checker.check_model(str(model_path))\n    session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])\n    del session\n    for path in Path(merged_dir).iterdir():\n        if path.suffix == ".json" or path.name in ("tokenizer.model", "vocab.json", "merges.txt", "chat_template.jinja"):\n            shutil.copy2(path, artifact_dir / path.name)\n    config_path = artifact_dir / "config.json"\n    config = json.loads(config_path.read_text(encoding="utf-8"))\n    config["transformers.js_config"] = {"use_external_data_format": False}\n    write_json(config_path, config)\n    return artifact_dir\n\n\ndef write_manifest(artifact_dir, lock, reports, gates, model_id):\n    artifact_dir = Path(artifact_dir)\n    if not re.fullmatch(r"[A-Za-z0-9._-]+/[A-Za-z0-9._-]+", model_id):\n        raise ValueError("model_id must have namespace/repository format")\n    files = [{"path": path.relative_to(artifact_dir).as_posix(), "size_bytes": path.stat().st_size,\n              "sha256": sha256_file(path)} for path in sorted(artifact_dir.rglob("*")) if path.is_file() and path.name != "runtime-manifest.json"]\n    manifest = {\n        "schema_version": 1, "runtime": "transformers.js", "model_id": model_id,\n        "base_model": BASE_MODEL, "base_revision": lock["base_revision"], "base_license": lock["base_license"],\n        "dtype": "q4", "device": "wasm", "max_input_tokens": MAX_INPUT_TOKENS, "max_new_tokens": MAX_NEW_TOKENS,\n        "model_files": [row["path"] for row in files if row["path"].startswith("onnx/")],\n        "quantization": {"method": "onnxruntime-matmul-4bits-rtn", "bits": 4, "block_size": 32,\n                         "activations": "fp32", "cache": "fp32", "accuracy_level": 0},\n        "files": files, "evaluation": reports, "release_gate": gates,\n        "status": "ready-for-manual-publication" if gates["release_eligible"] else "candidate-awaiting-validation",\n        "limitations": "GSAT style and CEEC coverage are approximations; the model does not certify exam difficulty. iPhone 13 must be tested separately.",\n    }\n    write_json(artifact_dir / "runtime-manifest.json", manifest)\n    return manifest\n\n\ndef write_upstream_notices(artifact_dir, lock):\n    """Preserve upstream card/notices from the SAME immutable model revision."""\n    from huggingface_hub import HfApi, hf_hub_download\n    import urllib.request\n    artifact_dir = Path(artifact_dir)\n    repo_files = HfApi().list_repo_files(BASE_MODEL, revision=lock["base_revision"])\n    notices = [name for name in repo_files if "/" not in name and\n               (name.upper().startswith("LICENSE") or name.upper().startswith("NOTICE"))]\n    for name in notices:\n        cached = hf_hub_download(BASE_MODEL, name, revision=lock["base_revision"])\n        shutil.copy2(cached, artifact_dir / name)\n    if not any(name.upper().startswith("LICENSE") for name in notices):\n        # Authoritative Apache license text, with normal HTTPS verification.\n        with urllib.request.urlopen("https://www.apache.org/licenses/LICENSE-2.0.txt", timeout=30) as response:\n            (artifact_dir / "LICENSE").write_bytes(response.read())\n    if "README.md" not in repo_files:\n        raise RuntimeError("Upstream model card missing; review model rights before distribution")\n    cached_card = hf_hub_download(BASE_MODEL, "README.md", revision=lock["base_revision"])\n    shutil.copy2(cached_card, artifact_dir / "UPSTREAM_MODEL_CARD.md")\n    (artifact_dir / "README.md").write_text(\n        "---\\nlicense: apache-2.0\\nlanguage:\\n- en\\nbase_model: " + BASE_MODEL +\n        "\\npipeline_tag: text-generation\\n---\\n\\n# GSAT example-sentence candidate\\n\\n" +\n        "Derived from " + BASE_MODEL + " at immutable revision `" + lock["base_revision"] + "`.\\n\\n" +\n        "This derivative adds a GSAT-style sentence-generation adapter and exports ONNX weight-only q4. " +\n        "The bundled tiny original dataset demonstrates the workflow; it does not establish efficacy. " +\n        "See runtime-manifest.json for release gates and evaluation reports. iPhone 13 validation is pending " +\n        "unless explicitly recorded by the operator. Retain LICENSE, any upstream NOTICE and UPSTREAM_MODEL_CARD.md.\\n\\n" +\n        "Training dataset license declarations: " + ", ".join(lock.get("dataset_licenses", [])) +\n        ". These declarations do not replace the operator\'s responsibility to verify rights.\\n",\n        encoding="utf-8")\n\n\ndef evaluate_q4(artifact_dir, tokenizer, records, curriculum, coverage, output_path, reference=None):\n    from optimum.onnxruntime import ORTModelForCausalLM\n    from transformers import AutoConfig, GenerationConfig\n    # Config/tokenizer live at artifact root; ONNX lives in its browser subfolder.\n    config = AutoConfig.from_pretrained(str(artifact_dir), trust_remote_code=False)\n    generation_config = GenerationConfig.from_pretrained(str(artifact_dir))\n    model = ORTModelForCausalLM.from_pretrained(str(artifact_dir), subfolder="onnx", file_name="model_q4.onnx",\n        config=config, generation_config=generation_config,\n        provider="CPUExecutionProvider", use_cache=True, use_io_binding=False)\n    return evaluate_model(model, tokenizer, records, curriculum, coverage, output_path, reference, device="cpu")\n\n\ndef package_candidate(artifact_dir, run_dir):\n    """Local download only. Publication is intentionally a separate manual action."""\n    for filename in ("run_lock.json", "evaluation_baseline.json", "evaluation_finetuned.json", "evaluation_q4.json"):\n        path = Path(run_dir) / filename\n        if path.exists():\n            shutil.copy2(path, Path(artifact_dir) / filename)\n    return shutil.make_archive(str(Path(run_dir) / "gsat-danube-q4-candidate"), "zip", artifact_dir)\n', 'export_curriculum.cjs': "'use strict';\n// Export the selected checkout's public learning data; never reads browser storage.\nconst fs = require('node:fs');\nconst path = require('node:path');\nconst vm = require('node:vm');\nconst root = path.resolve(__dirname, '..');\nconst context = vm.createContext({});\nfor (const file of ['vocabulary.js', 'collocations.js']) {\n  vm.runInContext(fs.readFileSync(path.join(root, 'assets/data', file), 'utf8'), context);\n}\nconst data = vm.runInContext(`({\n  schema_version: 1,\n  vocabulary: VOCABULARY.map(entry => ({\n    word: entry.word, level: entry.level, partOfSpeech: entry.partOfSpeech,\n    meaning: entry.meaning\n  })),\n  lexical_levels: {...CEEC_LEVEL_BY_WORD},\n  pattern_count: GSAT_COLLOCATION_GUIDE.reduce((sum, point) => sum + point.patterns.length, 0)\n})`, context);\nif (data.vocabulary.length !== 6012) throw new Error('Unexpected curriculum size; inspect source data.');\nconst destination = process.argv[2] || path.join(__dirname, 'data/gsat_curriculum.json');\nfs.mkdirSync(path.dirname(path.resolve(destination)), {recursive: true});\nfs.writeFileSync(destination, JSON.stringify(data, null, 2) + '\\n');\nconsole.log(`Exported ${data.vocabulary.length} vocabulary entries to ${destination}`);\n", 'gsat115_style_examples.jsonl': '{"id":"gsat115-style-001","target_word":"schedule","meaning_zh":"行程表","level":2,"target_forms":["schedule"],"sentence":"The interview was moved to Friday because the reporter could not fit another meeting into her schedule.","grammar_tag":"contextual_collocation","style_tag":"everyday_context","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-002","target_word":"vacancy","meaning_zh":"職缺","level":4,"target_forms":["vacancy"],"sentence":"Several experienced nurses applied for the vacancy after the hospital announced plans to expand its services.","grammar_tag":"contextual_collocation","style_tag":"everyday_context","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-003","target_word":"passion","meaning_zh":"熱情","level":3,"target_forms":["passion"],"sentence":"His passion for photography led him to record the changing streets of his hometown over several years.","grammar_tag":"contextual_collocation","style_tag":"historical_narrative","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-004","target_word":"routine","meaning_zh":"例行活動","level":3,"target_forms":["routine"],"sentence":"Adding a short walk to her daily routine helped the office worker feel less tired in the afternoon.","grammar_tag":"contextual_collocation","style_tag":"everyday_context","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-005","target_word":"benefit","meaning_zh":"好處","level":3,"target_forms":["benefit"],"sentence":"One benefit of growing vegetables at school is that students learn where their food comes from.","grammar_tag":"contextual_collocation","style_tag":"school_learning","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-006","target_word":"community","meaning_zh":"社區","level":4,"target_forms":["community"],"sentence":"The community library offers free classes so that older residents can learn to use digital services.","grammar_tag":"contextual_collocation","style_tag":"community_description","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-007","target_word":"simultaneous","meaning_zh":"同時的","level":6,"target_forms":["simultaneous"],"sentence":"The simultaneous arrival of several tour groups made it difficult for the small museum to provide enough guides.","grammar_tag":"contextual_collocation","style_tag":"everyday_context","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-008","target_word":"retain","meaning_zh":"記住；保留","level":4,"target_forms":["retain"],"sentence":"Students retain new expressions more easily when they use them in conversations rather than simply memorize a list.","grammar_tag":"adverbial_cause_time","style_tag":"school_learning","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-009","target_word":"risk","meaning_zh":"風險","level":3,"target_forms":["risk"],"sentence":"Because the trail becomes slippery after heavy rain, visitors are advised to avoid unnecessary risk.","grammar_tag":"adverbial_cause_time","style_tag":"everyday_context","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-010","target_word":"delay","meaning_zh":"延後","level":2,"target_forms":["delay"],"sentence":"The team decided to delay the outdoor experiment until the weather conditions became more stable.","grammar_tag":"adverbial_cause_time","style_tag":"science_explanation","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-011","target_word":"accommodate","meaning_zh":"容納","level":5,"target_forms":["accommodate"],"sentence":"As the number of visitors increased, the museum opened an additional entrance to accommodate larger groups.","grammar_tag":"adverbial_cause_time","style_tag":"community_description","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-012","target_word":"recover","meaning_zh":"恢復","level":2,"target_forms":["recover"],"sentence":"After resting in the shade for a while, the hikers began to recover from their exhausting climb.","grammar_tag":"adverbial_cause_time","style_tag":"historical_narrative","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-013","target_word":"prevent","meaning_zh":"防止","level":3,"target_forms":["prevent"],"sentence":"Since dry leaves can catch fire easily, campers should keep them away from cooking areas to prevent accidents.","grammar_tag":"adverbial_cause_time","style_tag":"everyday_context","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-014","target_word":"contrast","meaning_zh":"對比","level":4,"target_forms":["contrast"],"sentence":"Although the two festivals share a similar origin, their modern celebrations provide a sharp contrast.","grammar_tag":"concession_contrast","style_tag":"cultural_comparison","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-015","target_word":"vary","meaning_zh":"不同；變化","level":3,"target_forms":["vary"],"sentence":"While the basic ingredients remain the same, cooking methods may vary considerably from one region to another.","grammar_tag":"concession_contrast","style_tag":"cultural_comparison","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-016","target_word":"familiar","meaning_zh":"熟悉的","level":3,"target_forms":["familiar"],"sentence":"Even though the route looked familiar, the cyclists checked the map before turning into the narrow street.","grammar_tag":"concession_contrast","style_tag":"everyday_context","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-017","target_word":"preserve","meaning_zh":"保存","level":4,"target_forms":["preserve"],"sentence":"Although the old theater no longer hosts performances, local volunteers hope to preserve its original wooden seats.","grammar_tag":"concession_contrast","style_tag":"community_description","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-018","target_word":"individual","meaning_zh":"個人","level":2,"target_forms":["individual"],"sentence":"While an individual may prefer working alone, some tasks require members to share ideas and make decisions together.","grammar_tag":"concession_contrast","style_tag":"school_learning","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-019","target_word":"afford","meaning_zh":"負擔得起","level":3,"target_forms":["afford"],"sentence":"Although the new device includes several useful features, many students cannot afford it because of its high price.","grammar_tag":"concession_contrast","style_tag":"everyday_context","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-020","target_word":"skeptical","meaning_zh":"懷疑的","level":6,"target_forms":["skeptical"],"sentence":"Although the explanation sounded reasonable, several students remained skeptical until they could examine the results themselves.","grammar_tag":"concession_contrast","style_tag":"evidence_inference","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-021","target_word":"reliable","meaning_zh":"可靠的","level":3,"target_forms":["reliable"],"sentence":"The school invited a guide who could provide reliable information about the plants growing around the lake.","grammar_tag":"relative_clause","style_tag":"evidence_inference","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-022","target_word":"feature","meaning_zh":"特色","level":2,"target_forms":["feature"],"sentence":"The bridge, which connects two busy neighborhoods, has become a familiar feature of the city\'s riverside landscape.","grammar_tag":"relative_clause","style_tag":"community_description","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-023","target_word":"preference","meaning_zh":"偏好","level":5,"target_forms":["preference"],"sentence":"Customers whose preference is for lighter meals can choose dishes prepared with less oil and salt.","grammar_tag":"relative_clause","style_tag":"everyday_context","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-024","target_word":"evidence","meaning_zh":"證據","level":4,"target_forms":["evidence"],"sentence":"The marks that archaeologists found on the tools provided evidence of how people prepared food long ago.","grammar_tag":"relative_clause","style_tag":"evidence_inference","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-025","target_word":"retreat","meaning_zh":"撤退","level":4,"target_forms":["retreat"],"sentence":"Visitors followed a path that would allow them to retreat quickly if the storm moved closer.","grammar_tag":"relative_clause","style_tag":"historical_narrative","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-026","target_word":"unique","meaning_zh":"獨特的","level":3,"target_forms":["unique"],"sentence":"The village hosts a weekly market where travelers can buy unique products made by local artists.","grammar_tag":"relative_clause","style_tag":"community_description","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-027","target_word":"confirm","meaning_zh":"確認","level":3,"target_forms":["confirm","confirmed"],"sentence":"The location of the missing painting was confirmed after museum workers compared the photograph with their old records.","grammar_tag":"passive_voice","style_tag":"evidence_inference","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-028","target_word":"perceive","meaning_zh":"視為；認為","level":5,"target_forms":["perceive","perceived"],"sentence":"In some workplaces, asking questions is perceived as a sign of interest rather than a lack of knowledge.","grammar_tag":"passive_voice","style_tag":"cultural_comparison","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-029","target_word":"adjust","meaning_zh":"調整","level":4,"target_forms":["adjust","adjusted"],"sentence":"Each reading plan is adjusted to match the learner\'s interests, available time, and current vocabulary knowledge.","grammar_tag":"passive_voice","style_tag":"school_learning","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-030","target_word":"associate","meaning_zh":"聯想；連結","level":4,"target_forms":["associate","associated"],"sentence":"The festival is often associated with family gatherings, although many young people now celebrate it with friends.","grammar_tag":"passive_voice","style_tag":"cultural_comparison","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-031","target_word":"divide","meaning_zh":"分成","level":2,"target_forms":["divide","divided"],"sentence":"Participants were divided into small groups so that everyone had more opportunities to practice speaking.","grammar_tag":"passive_voice","style_tag":"school_learning","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-032","target_word":"restore","meaning_zh":"修復","level":4,"target_forms":["restore","restored"],"sentence":"The damaged garden was restored by volunteers who planted trees and repaired the paths after the storm.","grammar_tag":"passive_voice","style_tag":"community_description","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-033","target_word":"inhabit","meaning_zh":"居住於","level":6,"target_forms":["inhabit","inhabited"],"sentence":"The island is inhabited by families who depend on fishing, although younger residents increasingly seek jobs in nearby cities.","grammar_tag":"passive_voice","style_tag":"community_description","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-034","target_word":"design","meaning_zh":"設計","level":2,"target_forms":["design","designed"],"sentence":"Designed for crowded classrooms, the folding desk can be stored easily when students need more space for group activities.","grammar_tag":"participial_modifier","style_tag":"school_learning","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-035","target_word":"locate","meaning_zh":"位於","level":3,"target_forms":["locate","located"],"sentence":"Located beside an old railway station, the café attracts travelers who want to learn about the town\'s history.","grammar_tag":"participial_modifier","style_tag":"community_description","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-036","target_word":"concern","meaning_zh":"使擔憂","level":2,"target_forms":["concern","concerned"],"sentence":"Concerned about rising electricity costs, the school installed lights that switch off automatically when rooms are empty.","grammar_tag":"participial_modifier","style_tag":"community_description","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-037","target_word":"range","meaning_zh":"範圍涵蓋","level":2,"target_forms":["range","ranging"],"sentence":"The exhibition displays everyday objects ranging from handmade baskets to early radios, showing how family life has changed.","grammar_tag":"participial_modifier","style_tag":"historical_narrative","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-038","target_word":"lead","meaning_zh":"導致；引導","level":1,"target_forms":["lead","leading"],"sentence":"The researchers observed insects entering the covered container, leading them to examine whether light affected the insects\' behavior.","grammar_tag":"participial_modifier","style_tag":"evidence_inference","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-039","target_word":"emerge","meaning_zh":"出現","level":4,"target_forms":["emerge","emerging"],"sentence":"With new shops emerging near the harbor, the district has become a popular destination for weekend visitors.","grammar_tag":"participial_modifier","style_tag":"community_description","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-040","target_word":"suggest","meaning_zh":"顯示；暗示","level":2,"target_forms":["suggest","suggests"],"sentence":"The survey suggests that students may read more willingly when they are allowed to choose books that interest them.","grammar_tag":"noun_clause_reporting","style_tag":"evidence_inference","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-041","target_word":"indicate","meaning_zh":"顯示","level":2,"target_forms":["indicate"],"sentence":"A change in the birds\' feeding pattern may indicate that the weather is about to become colder.","grammar_tag":"noun_clause_reporting","style_tag":"evidence_inference","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-042","target_word":"discover","meaning_zh":"發現","level":2,"target_forms":["discover","discovered"],"sentence":"The students discovered that soil under the trees remained cooler than soil exposed to direct sunlight.","grammar_tag":"noun_clause_reporting","style_tag":"science_explanation","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-043","target_word":"explain","meaning_zh":"解釋","level":1,"target_forms":["explain","explained"],"sentence":"The guide explained why the houses had thick walls and how those walls helped keep the rooms cool.","grammar_tag":"noun_clause_reporting","style_tag":"historical_narrative","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-044","target_word":"assume","meaning_zh":"假設；認為","level":3,"target_forms":["assume"],"sentence":"Some visitors assume that every old building is a museum, but many of these houses are still occupied.","grammar_tag":"noun_clause_reporting","style_tag":"cultural_comparison","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-045","target_word":"report","meaning_zh":"報告；通報","level":1,"target_forms":["report","reported"],"sentence":"Residents reported that the river looked clearer after the town improved the system for treating household waste.","grammar_tag":"noun_clause_reporting","style_tag":"evidence_inference","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-046","target_word":"comprehend","meaning_zh":"理解","level":5,"target_forms":["comprehend"],"sentence":"The students could not comprehend why the same experiment produced different results when they moved the plants to another room.","grammar_tag":"noun_clause_reporting","style_tag":"science_explanation","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-047","target_word":"provide","meaning_zh":"提供","level":2,"target_forms":["provide"],"sentence":"The school planted trees near the playground to provide shade for children during the hottest part of the day.","grammar_tag":"purpose_result","style_tag":"school_learning","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-048","target_word":"reduce","meaning_zh":"減少","level":3,"target_forms":["reduce"],"sentence":"The city introduced a bicycle program to reduce traffic and make short journeys more convenient.","grammar_tag":"purpose_result","style_tag":"community_description","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-049","target_word":"allow","meaning_zh":"讓；允許","level":1,"target_forms":["allow"],"sentence":"The museum offers audio descriptions to allow visitors with poor eyesight to explore the exhibits more independently.","grammar_tag":"purpose_result","style_tag":"community_description","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-050","target_word":"enable","meaning_zh":"使能夠","level":3,"target_forms":["enable"],"sentence":"Clear labels on recycling bins enable people to sort their waste correctly without asking staff for help.","grammar_tag":"purpose_result","style_tag":"everyday_context","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-051","target_word":"contribute","meaning_zh":"捐助；貢獻","level":4,"target_forms":["contribute"],"sentence":"Local businesses contribute money to the project so that the youth center can offer free weekend classes.","grammar_tag":"purpose_result","style_tag":"community_description","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-052","target_word":"limit","meaning_zh":"限制","level":2,"target_forms":["limit","limited"],"sentence":"The number of seats was limited so that visitors could move around the small hall more comfortably.","grammar_tag":"purpose_result","style_tag":"everyday_context","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-053","target_word":"distraction","meaning_zh":"分心；干擾","level":6,"target_forms":["distraction"],"sentence":"To avoid distraction, students turned off their phones before discussing the results of the science experiment.","grammar_tag":"purpose_result","style_tag":"school_learning","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-054","target_word":"significant","meaning_zh":"顯著的","level":3,"target_forms":["significant"],"sentence":"The difference in water use was more significant than the students had expected when they began the project.","grammar_tag":"comparison_degree","style_tag":"science_explanation","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-055","target_word":"effective","meaning_zh":"有效的","level":2,"target_forms":["effective"],"sentence":"For some learners, explaining an idea to a partner is more effective than copying the same notes several times.","grammar_tag":"comparison_degree","style_tag":"school_learning","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-056","target_word":"frequent","meaning_zh":"頻繁的","level":3,"target_forms":["frequent"],"sentence":"As summer approached, afternoon showers became more frequent, making it harder for the workers to finish the outdoor repairs.","grammar_tag":"comparison_degree","style_tag":"historical_narrative","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-057","target_word":"minor","meaning_zh":"微小的","level":2,"target_forms":["minor"],"sentence":"What seemed a minor change in the schedule created a much greater problem than the organizers had predicted.","grammar_tag":"comparison_degree","style_tag":"everyday_context","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-058","target_word":"complicate","meaning_zh":"使複雜","level":4,"target_forms":["complicate","complicated"],"sentence":"Unexpected safety rules complicated the library project far more than the organizers had anticipated when they purchased the land.","grammar_tag":"comparison_degree","style_tag":"everyday_context","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-059","target_word":"similar","meaning_zh":"相似的","level":2,"target_forms":["similar"],"sentence":"The two paintings are similar in color, but the newer work gives a much stronger impression of movement.","grammar_tag":"comparison_degree","style_tag":"cultural_comparison","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-060","target_word":"responsible","meaning_zh":"負責的","level":2,"target_forms":["responsible"],"sentence":"If each visitor is responsible for taking away their own trash, the park will be easier to keep clean.","grammar_tag":"conditional_modal","style_tag":"everyday_context","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-061","target_word":"survive","meaning_zh":"存活","level":2,"target_forms":["survive"],"sentence":"Some plants can survive long periods without rain if their roots can reach water deep in the soil.","grammar_tag":"conditional_modal","style_tag":"science_explanation","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-062","target_word":"respond","meaning_zh":"反應","level":2,"target_forms":["respond"],"sentence":"A driver may have less time to respond if a child suddenly runs into the road.","grammar_tag":"conditional_modal","style_tag":"everyday_context","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-063","target_word":"likely","meaning_zh":"可能的","level":2,"target_forms":["likely"],"sentence":"Without regular practice, students are less likely to remember expressions they have encountered only once.","grammar_tag":"conditional_modal","style_tag":"school_learning","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-064","target_word":"avoid","meaning_zh":"避免","level":2,"target_forms":["avoid"],"sentence":"Unless the damaged wires are replaced, the workers cannot avoid the risk of another power failure.","grammar_tag":"conditional_modal","style_tag":"everyday_context","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-065","target_word":"opportunity","meaning_zh":"機會","level":3,"target_forms":["opportunity"],"sentence":"If the weather remains calm, the team will have an opportunity to observe the island\'s wildlife more closely.","grammar_tag":"conditional_modal","style_tag":"science_explanation","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-066","target_word":"furthermore","meaning_zh":"此外","level":4,"target_forms":["furthermore"],"sentence":"The new reading room is close to the station; furthermore, it remains open late enough for evening students.","grammar_tag":"discourse_linking","style_tag":"school_learning","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-067","target_word":"however","meaning_zh":"然而","level":1,"target_forms":["however"],"sentence":"The beach looked clean from a distance; however, volunteers found small pieces of plastic buried in the sand.","grammar_tag":"discourse_linking","style_tag":"evidence_inference","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-068","target_word":"therefore","meaning_zh":"因此","level":2,"target_forms":["therefore"],"sentence":"The village received more visitors than expected; therefore, local buses began running more frequently during the holiday.","grammar_tag":"discourse_linking","style_tag":"community_description","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-069","target_word":"meanwhile","meaning_zh":"同時","level":3,"target_forms":["meanwhile"],"sentence":"The engineers tested the new bridge; meanwhile, residents continued to use the older road on the other side of town.","grammar_tag":"discourse_linking","style_tag":"historical_narrative","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-070","target_word":"initial","meaning_zh":"最初的","level":4,"target_forms":["initial"],"sentence":"The team abandoned its initial plan; instead, it examined how birds responded to the noise from a nearby road.","grammar_tag":"discourse_linking","style_tag":"school_learning","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-071","target_word":"ultimate","meaning_zh":"最終的","level":5,"target_forms":["ultimate"],"sentence":"Several playground designs were considered; in the end, the ultimate choice depended on the needs of local children.","grammar_tag":"discourse_linking","style_tag":"community_description","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-072","target_word":"increase","meaning_zh":"增加","level":2,"target_forms":["increase","increased"],"sentence":"The number of people using the public library has increased since it began offering free access to online courses.","grammar_tag":"perfect_tense","style_tag":"school_learning","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-073","target_word":"establish","meaning_zh":"設立","level":4,"target_forms":["establish","established"],"sentence":"By the time the new road opened, several farmers had established small shops to sell their products to travelers.","grammar_tag":"perfect_tense","style_tag":"historical_narrative","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-074","target_word":"adapt","meaning_zh":"調整以適應","level":4,"target_forms":["adapt","adapted"],"sentence":"Many local shops have adapted their services as more customers have begun ordering daily necessities through their phones.","grammar_tag":"perfect_tense","style_tag":"everyday_context","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-075","target_word":"develop","meaning_zh":"制定；發展","level":2,"target_forms":["develop","developed"],"sentence":"The club had developed a simple safety plan before it invited younger students to join the mountain trip.","grammar_tag":"perfect_tense","style_tag":"school_learning","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-076","target_word":"remain","meaning_zh":"仍然是","level":3,"target_forms":["remain","remained"],"sentence":"Although the town has changed rapidly, the old marketplace has remained an important meeting place for local families.","grammar_tag":"perfect_tense","style_tag":"community_description","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-077","target_word":"replace","meaning_zh":"取代","level":3,"target_forms":["replace","replaced"],"sentence":"Several printed notices have been replaced with digital screens, allowing station staff to update travel information more quickly.","grammar_tag":"perfect_tense","style_tag":"community_description","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n{"id":"gsat115-style-078","target_word":"underestimate","meaning_zh":"低估","level":6,"target_forms":["underestimate","underestimated"],"sentence":"The travelers had underestimated the distance between villages before they realized how slowly they were moving along the mountain path.","grammar_tag":"perfect_tense","style_tag":"historical_narrative","source":"self-authored-gsat115-style","license":"CC0-1.0"}\n', 'gsat_examples_demo.jsonl': '{"id": "demo-001", "target_word": "challenge", "sentence": "Learning to speak in public is a challenge that helps students grow.", "meaning_zh": "挑戰", "level": 2, "target_forms": ["challenge"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-002", "target_word": "protect", "sentence": "Our school plants trees near the river to protect the local environment.", "meaning_zh": "保護", "level": 2, "target_forms": ["protect"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-003", "target_word": "reduce", "sentence": "Students can reduce plastic waste by bringing their own bottles to school.", "meaning_zh": "減少", "level": 3, "target_forms": ["reduce"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-004", "target_word": "research", "sentence": "The students compared several reports before they began their research on clean water.", "meaning_zh": "研究", "level": 4, "target_forms": ["research"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-005", "target_word": "evidence", "sentence": "The team collected evidence to explain why the river became polluted last summer.", "meaning_zh": "證據", "level": 4, "target_forms": ["evidence"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-006", "target_word": "benefit", "sentence": "One benefit of reading every day is learning how to express ideas clearly.", "meaning_zh": "好處", "level": 3, "target_forms": ["benefit"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-007", "target_word": "improve", "sentence": "Keeping a short reading diary can improve your ability to organize new ideas.", "meaning_zh": "改善", "level": 2, "target_forms": ["improve"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-008", "target_word": "develop", "sentence": "Young people develop useful skills when they work together on community projects.", "meaning_zh": "發展", "level": 2, "target_forms": ["develop"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-009", "target_word": "provide", "sentence": "The library will provide quiet spaces for students during the final examination week.", "meaning_zh": "提供", "level": 2, "target_forms": ["provide"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-010", "target_word": "prevent", "sentence": "Wearing a helmet can prevent serious injuries when you ride a bicycle.", "meaning_zh": "防止", "level": 3, "target_forms": ["prevent"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-011", "target_word": "responsible", "sentence": "Everyone in our class is responsible for keeping the classroom clean and safe.", "meaning_zh": "負責的", "level": 2, "target_forms": ["responsible"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-012", "target_word": "concern", "sentence": "Water pollution is a growing concern for people who live near the river.", "meaning_zh": "擔憂", "level": 2, "target_forms": ["concern"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-013", "target_word": "effect", "sentence": "The report explains the effect of air pollution on children in large cities.", "meaning_zh": "影響", "level": 2, "target_forms": ["effect"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-014", "target_word": "environment", "sentence": "Small changes in our daily habits can help create a cleaner environment.", "meaning_zh": "環境", "level": 2, "target_forms": ["environment"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-015", "target_word": "pollution", "sentence": "The town built a new system to reduce water pollution from nearby factories.", "meaning_zh": "污染", "level": 3, "target_forms": ["pollution"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-016", "target_word": "community", "sentence": "People in the community worked together to repair the playground after the storm.", "meaning_zh": "社區", "level": 4, "target_forms": ["community"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-017", "target_word": "volunteer", "sentence": "A volunteer helped the children choose books at the local library on Saturday.", "meaning_zh": "志工", "level": 4, "target_forms": ["volunteer"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-018", "target_word": "opportunity", "sentence": "Joining the school club gave me an opportunity to meet students from other classes.", "meaning_zh": "機會", "level": 3, "target_forms": ["opportunity"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-019", "target_word": "experience", "sentence": "Her experience of living abroad taught her to respect different ways of life.", "meaning_zh": "經驗", "level": 1, "target_forms": ["experience"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-020", "target_word": "education", "sentence": "Access to education can help children build a better future for their families.", "meaning_zh": "教育", "level": 2, "target_forms": ["education"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-021", "target_word": "technology", "sentence": "New technology allows students in distant villages to attend lessons from home.", "meaning_zh": "科技", "level": 2, "target_forms": ["technology"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-022", "target_word": "information", "sentence": "Check the source of the information before you share it with your friends.", "meaning_zh": "資訊", "level": 3, "target_forms": ["information"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-023", "target_word": "compare", "sentence": "We need to compare the two plans before choosing the best way forward.", "meaning_zh": "比較", "level": 2, "target_forms": ["compare"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-024", "target_word": "result", "sentence": "The result of the study suggests that regular exercise can improve sleep.", "meaning_zh": "結果", "level": 2, "target_forms": ["result"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-025", "target_word": "method", "sentence": "The teacher showed us a simple method for remembering the new words.", "meaning_zh": "方法", "level": 2, "target_forms": ["method"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-026", "target_word": "solution", "sentence": "Students discussed a possible solution to the traffic problem near their school.", "meaning_zh": "解決方法", "level": 2, "target_forms": ["solution"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-027", "target_word": "decision", "sentence": "Before making a decision, consider how your choice may affect other people.", "meaning_zh": "決定", "level": 2, "target_forms": ["decision"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-028", "target_word": "purpose", "sentence": "The purpose of the project is to help students understand local history.", "meaning_zh": "目的", "level": 2, "target_forms": ["purpose"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-029", "target_word": "support", "sentence": "Our classmates offered support when we felt worried about the coming examination.", "meaning_zh": "支持", "level": 2, "target_forms": ["support"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-030", "target_word": "encourage", "sentence": "Teachers encourage students to ask questions when they do not understand the lesson.", "meaning_zh": "鼓勵", "level": 2, "target_forms": ["encourage"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-031", "target_word": "contribute", "sentence": "Each student can contribute to the project by sharing a useful idea.", "meaning_zh": "貢獻", "level": 4, "target_forms": ["contribute"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-032", "target_word": "participate", "sentence": "Students who participate in group discussions often learn to explain their opinions clearly.", "meaning_zh": "參與", "level": 2, "target_forms": ["participate"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-033", "target_word": "depend", "sentence": "Many small farms depend on regular rain to produce enough food each year.", "meaning_zh": "依賴", "level": 2, "target_forms": ["depend"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-034", "target_word": "adapt", "sentence": "New students may need time to adapt to life in a different school.", "meaning_zh": "適應", "level": 4, "target_forms": ["adapt"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-035", "target_word": "aware", "sentence": "People are becoming aware of the damage that plastic waste causes in the ocean.", "meaning_zh": "知道的", "level": 3, "target_forms": ["aware"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-036", "target_word": "available", "sentence": "Free drinking water is available to everyone who visits the community center.", "meaning_zh": "可取得的", "level": 2, "target_forms": ["available"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-037", "target_word": "satisfy", "sentence": "The students were satisfied with the results of their first science project.", "meaning_zh": "使滿意", "level": 2, "target_forms": ["satisfied"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-038", "target_word": "capable", "sentence": "She is capable of explaining difficult ideas in a clear and simple way.", "meaning_zh": "有能力的", "level": 3, "target_forms": ["capable"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-039", "target_word": "quality", "sentence": "The quality of the water improved after the factory changed its waste system.", "meaning_zh": "品質", "level": 2, "target_forms": ["quality"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-040", "target_word": "increase", "sentence": "Regular reading can increase your knowledge and help you understand different points of view.", "meaning_zh": "增加", "level": 2, "target_forms": ["increase"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-041", "target_word": "decrease", "sentence": "The number of traffic accidents began to decrease after the town improved its roads.", "meaning_zh": "減少", "level": 3, "target_forms": ["decrease"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-042", "target_word": "essential", "sentence": "Clean water is essential for the health of every person in the community.", "meaning_zh": "必要的", "level": 4, "target_forms": ["essential"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-043", "target_word": "achieve", "sentence": "Students can achieve their goals by making careful plans and practicing every day.", "meaning_zh": "達成", "level": 3, "target_forms": ["achieve"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-044", "target_word": "balance", "sentence": "It is important to find a balance between schoolwork and time for rest.", "meaning_zh": "平衡", "level": 2, "target_forms": ["balance"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-045", "target_word": "beneficial", "sentence": "Spending time outdoors can be beneficial to students who feel tired after school.", "meaning_zh": "有益的", "level": 5, "target_forms": ["beneficial"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-046", "target_word": "effective", "sentence": "The class found an effective way to collect and recycle used paper.", "meaning_zh": "有效的", "level": 2, "target_forms": ["effective"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-047", "target_word": "influence", "sentence": "A good teacher can influence how students think about their future careers.", "meaning_zh": "影響", "level": 2, "target_forms": ["influence"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-048", "target_word": "observe", "sentence": "Students observe the plants each morning and record any changes in their growth.", "meaning_zh": "觀察", "level": 3, "target_forms": ["observe"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-049", "target_word": "perspective", "sentence": "Reading stories from other cultures helps us see life from a different perspective.", "meaning_zh": "觀點", "level": 5, "target_forms": ["perspective"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-050", "target_word": "recommend", "sentence": "Doctors recommend regular exercise to help people stay healthy as they grow older.", "meaning_zh": "建議", "level": 5, "target_forms": ["recommend"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-051", "target_word": "sustainable", "sentence": "The city plans to build a more sustainable transport system with electric buses.", "meaning_zh": "永續的", "level": 5, "target_forms": ["sustainable"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-052", "target_word": "significant", "sentence": "The new reading program made a significant difference to the students who joined it.", "meaning_zh": "顯著的", "level": 3, "target_forms": ["significant"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-053", "target_word": "resource", "sentence": "Water is a valuable resource that we should use carefully in our daily lives.", "meaning_zh": "資源", "level": 3, "target_forms": ["resource"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-054", "target_word": "respond", "sentence": "The students learned how to respond calmly when someone disagreed with their ideas.", "meaning_zh": "回應", "level": 2, "target_forms": ["respond"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-055", "target_word": "recognize", "sentence": "It can be difficult to recognize a false story when it looks like real news.", "meaning_zh": "認出", "level": 3, "target_forms": ["recognize"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-056", "target_word": "maintain", "sentence": "Schools should maintain safe spaces where students can learn and share their opinions.", "meaning_zh": "維持", "level": 2, "target_forms": ["maintain"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-057", "target_word": "analyze", "sentence": "The class will analyze the survey results before writing a report about student habits.", "meaning_zh": "分析", "level": 4, "target_forms": ["analyze"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-058", "target_word": "consider", "sentence": "We should consider the needs of older people when designing public spaces.", "meaning_zh": "考慮", "level": 2, "target_forms": ["consider"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-059", "target_word": "respect", "sentence": "Learning about other cultures can help us respect people whose lives are different.", "meaning_zh": "尊重", "level": 2, "target_forms": ["respect"], "source": "self-authored-demo", "license": "CC0-1.0"}\n{"id": "demo-060", "target_word": "approach", "sentence": "The teacher used a different approach to help students understand the difficult passage.", "meaning_zh": "方法", "level": 2, "target_forms": ["approach"], "source": "self-authored-demo", "license": "CC0-1.0"}\n'}

HELPER_DIR = RUN_DIR / 'helpers'
HELPER_DIR.mkdir()
for filename, source in EMBEDDED_FILES.items():
    (HELPER_DIR / filename).write_text(source, encoding='utf-8')
sys.path.insert(0, str(HELPER_DIR))
import gsat_workflow as workflow
subprocess.run([sys.executable, '-m', 'pip', 'install', *workflow.PACKAGES], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=False)
import torch
gpu = workflow.gpu_guard()
print('GPU:', gpu['device'], 'VRAM GiB:', gpu['vram_gib'], '運算精度:', gpu['dtype'])


## 2. 固定網站課綱來源、載入原創資料

讀取網站公開的 6,012 筆單字與 CEEC 級別；來源固定至已檢查的 `restore-old-site` commit。按 `target_word` 決定 train/validation/test，所有同字例句留在同一區。**不把驗證或測試例句放入訓練**。示範資料過小，正式發布門檻會不通過；新增 PDF 參考不會增加原創訓練例句數。

`UPLOAD_REFERENCE_PDF=True` 時可一次選取 111、112、113、114、115 試卷與 115 學年度起適用考試說明，也可只選其中一份。每份需能抽取至少一個英文八詞片段；中文說明頁可以沒有英文片段。按 SHA-256 去除相同檔案內容，每頁分別取片段再合併，避免跨頁湊句。`run_lock.json` 只保存檔名、雜湊、頁數與片段數，不保存試卷或抽取文字。未上傳代表防照抄比對尚未執行；上傳子集也不代表五年覆蓋完整。

自備 JSONL 欄位：`id,target_word,sentence,level,source,license,target_forms,meaning_zh,grammar_tag`。`source/license` 必須真實，避免個資和未獲授權句子；13 種 grammar tag 與網站選單一致。長輸入與長答案會直接報錯，不會截斷成不完整訓練句。


In [ ]:
subprocess.run(['git', 'init', '-q', str(SOURCE_DIR)], check=True)
subprocess.run(['git', '-C', str(SOURCE_DIR), 'remote', 'add', 'origin', 'https://github.com/jasoart/E.git'], check=True)
subprocess.run(['git', '-C', str(SOURCE_DIR), 'fetch', '--depth', '1', 'origin', BASE_REPO_SHA], check=True)
subprocess.run(['git', '-C', str(SOURCE_DIR), 'checkout', '--detach', '-q', 'FETCH_HEAD'], check=True)
source_sha = subprocess.check_output(['git', '-C', str(SOURCE_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert source_sha == BASE_REPO_SHA, '課綱來源commit不一致'
source_colab = SOURCE_DIR / 'colab'
source_colab.mkdir(exist_ok=True)
(source_colab / 'export_curriculum.cjs').write_text(EMBEDDED_FILES['export_curriculum.cjs'], encoding='utf-8')
if not shutil.which('node'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'nodejs'], check=True)
curriculum_path = RUN_DIR / 'gsat_curriculum.json'
subprocess.run(['node', str(source_colab / 'export_curriculum.cjs'), str(curriculum_path)], check=True)
curriculum_document = json.loads(curriculum_path.read_text(encoding='utf-8'))
curriculum = workflow.curriculum_index(curriculum_document)
coverage = workflow.build_coverage_index(curriculum, curriculum_document['lexical_levels'])
dataset_paths = [HELPER_DIR / 'gsat115_style_examples.jsonl']
if ADD_GENERIC_DEMO:
    dataset_paths.append(HELPER_DIR / 'gsat_examples_demo.jsonl')
if UPLOAD_MY_JSONL:
    uploaded = files.upload()
    dataset_paths = []
    for filename, content in uploaded.items():
        if not filename.lower().endswith('.jsonl'):
            raise ValueError('請只上傳JSONL資料集')
        destination = RUN_DIR / Path(filename).name
        destination.write_bytes(content)
        dataset_paths.append(destination)
    if not dataset_paths:
        raise ValueError('沒有上傳JSONL資料集')
records = [row for path in dataset_paths for row in workflow.read_jsonl(path)]
if len({row['id'] for row in records}) != len(records):
    raise ValueError('跨資料集id重複')
partitions = workflow.split_records(records)
if not all(partitions.values()):
    raise ValueError('至少一個分割區沒有資料；請增加不同目標單字')
for name, rows in partitions.items():
    print(name, '例句:', len(rows), '目標詞:', len({r['target_word'] for r in rows}))
if not DEMO_MODE and len({r['target_word'] for r in partitions['train']}) < 200:
    raise ValueError('正式模式需至少200個訓練目標詞；目前僅適合示範')
reference = None
reference_metadata = None
if UPLOAD_REFERENCE_PDF:
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('沒有上傳參考PDF；請選取至少一份')
    if any(not name.lower().endswith('.pdf') for name in uploaded):
        raise ValueError('參考檔案請全部使用PDF，不接受混合其他檔案')
    if any(not content for content in uploaded.values()):
        raise ValueError('參考PDF不得為空檔案')
    filenames = [Path(name).name for name in uploaded]
    if len(set(filenames)) != len(filenames):
        raise ValueError('參考PDF檔名重複；請使用不同檔名')
    reference_dir = RUN_DIR / 'reference_pdfs'
    reference_dir.mkdir(exist_ok=True)
    pdf_paths = []
    for name, content in uploaded.items():
        pdf_path = reference_dir / Path(name).name
        if pdf_path.exists() and pdf_path.read_bytes() != content:
            raise ValueError(f'參考檔案已存在且內容不同：{pdf_path.name}')
        pdf_path.write_bytes(content)
        pdf_paths.append(pdf_path)
    reference_index = workflow.reference_ngram_index(pdf_paths)
    reference = reference_index['ngrams']
    reference_metadata = reference_index['metadata']
    print('上傳PDF:', reference_metadata['uploaded_document_count'],
          '去重後:', reference_metadata['unique_document_count'],
          '頁數:', reference_metadata['page_count'], '英文八詞片段:', len(reference),
          '（僅防照抄比對，不加入訓練／ZIP）')
print('示範prompt:', workflow.build_messages(partitions['train'][0], curriculum))


## 3. 記錄實際模型版本與原始基準

公開模型正常下載不需要 Token。使用既有 Hugging Face 認證；不印出 Token。此格讀取真正的模型 metadata、Apache-2.0 授權、`LlamaForCausalLM` 架構與 immutable commit，若與預期不同就停止，避免假設不符仍繼續匯出。

原始模型、微調模型、q4 模型都使用相同測試 prompt 與貪婪生成設定（與網站預設相同）。基準是未加入 adapter 的完整精度模型；QLoRA 不會把自己的 NF4 初始化當成原始基準。


In [ ]:
lock = workflow.lock_model_revision(RUN_DIR, dataset_paths)
lock.update(source_repository='jasoart/E', source_repository_revision=BASE_REPO_SHA,
            method=METHOD, initialization=INITIALIZATION, demo_mode=DEMO_MODE,
            dataset_licenses=sorted({r['license'] for r in records}),
            reference_index=reference_metadata)
workflow.write_json(RUN_DIR / 'run_lock.json', lock)
reports = {}
baseline, tokenizer = workflow.load_baseline_model(lock)
reports['baseline'] = workflow.evaluate_model(baseline, tokenizer, partitions['test'], curriculum,
    coverage, RUN_DIR / 'evaluation_baseline.json', reference, device='cuda')
print('原始基準:', reports['baseline'])
del baseline
gc.collect(); torch.cuda.empty_cache()


## 4. Assistant-only LoRA 微調並與基準比較

只對答案與 EOS 算 loss，prompt 和 padding 的 labels 都是 `-100`。預設 `r=16`、`alpha=32`、`all-linear`、20 示範步；不訓練網站對話歷史。PiSSA 會保留初始 adapter 以轉成可載回原始模型的 LoRA；LoftQ 使用 `alpha=r=16` 避免縮放兩倍 residual，並僅使用 PEFT 官方 one-step replacement，與完整論文演算法不同。


In [ ]:
model, tokenizer, gpu = workflow.load_training_model(lock, METHOD, INITIALIZATION, output_dir=RUN_DIR)
adapter_path = workflow.train_adapter(model, tokenizer, partitions, curriculum, RUN_DIR, gpu,
    demo=DEMO_MODE, initialization=INITIALIZATION)
model.config.use_cache = True
reports['finetuned'] = workflow.evaluate_model(model, tokenizer, partitions['test'], curriculum,
    coverage, RUN_DIR / 'evaluation_finetuned.json', reference, device='cuda')
print('微調模型:', reports['finetuned'])
del model
gc.collect(); torch.cuda.empty_cache()


## 5. 重載原始 fp32、合併、ONNX with-past、4-bit weight-only

匯出與量化在 CPU 進行，需要時間、記憶體與磁碟；Colab 小型 VM 若記憶體不足，先停止其他工作或改用較多 RAM 的執行環境。**不要把 NF4/bitsandbytes 權重直接放網站**。保留 fp32 activations/KV cache；短 context 有助手機記憶體控制。

Optimum 自動比較原始與 ONNX 數值；檢查 `input_ids/attention_mask/position_ids/past_key_values` 與 `present`，再產生 `onnx/model_q4.onnx`。所有 tensors 內嵌，避免 browser 外部 shard 名稱錯配；超過 2 GiB 直接停止。q4 可以縮減 linear weights，但 embedding/tokenizer/cache 仍占空間，實際大小以輸出檔為準。


In [ ]:
merged_dir = RUN_DIR / 'merged_fp32'
merged_model = workflow.merge_adapter(lock, adapter_path, merged_dir)
del merged_model
gc.collect()
artifact_dir = workflow.export_q4(merged_dir, RUN_DIR / 'browser_model')
reports['q4'] = workflow.evaluate_q4(artifact_dir, tokenizer, partitions['test'], curriculum,
    coverage, RUN_DIR / 'evaluation_q4.json', reference)
workflow.write_upstream_notices(artifact_dir, lock)
print('部署q4模型:', reports['q4'])


## 6. 品質門檻、候選 ZIP 與網站設定

門檻：至少 200 個訓練詞、50 個獨立測試詞；目標詞/長度 ≥90%、單句/CEEC 字彙涵蓋 ≥95%；量化下降不超過 2 個百分點；上傳參考 PDF 後八詞重複率為 0；模型 ≤768 MiB；人工至少 50 句文法/詞義正確率 ≥95%；iPhone 13 實測完成。這些是本專案保守門檻，並非 CEEC 認證標準。

demo 或手機/人工尚未測試會留下 `candidate-awaiting-validation`。仍可下載候選 artifact 測試，但不應當成正式模型。`evaluation_*.json` 保存輸出方便人工評閱；CSV/指標不取代教師判斷。

ZIP 解壓後，透過自己的 HTTPS 網站目錄或人工上傳至 Hugging Face 公開模型庫提供檔案；保留 `runtime-manifest.json/config.json/tokenizer*/generation_config.json/onnx/model_q4.onnx` 的目錄結構與自動保留的原模型授權及來源。前端詞卡的 **學測例句生成 → 設定 Colab 匯出的模型** 輸入公開 `owner/model` 或同網站 `/models/gsat-danube/`，使用者按載入後才下載。首次下載可能數百 MB，先使用 Wi-Fi。

iPhone 13 Safari 驗證：HTTPS 載入、10 個不同詞與各文法類型、記憶體不重載、首 token/總時間、取消/重試、快取後離線重開（仍需網站/JS 已快取），並核對每句文法和詞義。WebGL、WebGPU 和原生 Core ML 不在本預設路徑。真機完成前不能保證手機速度或離線能力。


In [ ]:
# 先把本地評估與鎖檔加入artifact，再計算manifest逐檔SHA256。
for name in ('run_lock.json', 'evaluation_baseline.json', 'evaluation_finetuned.json', 'evaluation_q4.json'):
    shutil.copy2(RUN_DIR / name, artifact_dir / name)
artifact_bytes = sum(path.stat().st_size for path in artifact_dir.rglob('*') if path.is_file())
gates = workflow.release_gate(partitions, reports, artifact_bytes, HUMAN_REVIEW, IPHONE_REVIEW)
manifest = workflow.write_manifest(artifact_dir, lock, reports, gates, MODEL_ID)
print('Artifact MiB:', round(artifact_bytes / 1024**2, 1))
print(json.dumps(gates, ensure_ascii=False, indent=2))
zip_path = workflow.package_candidate(artifact_dir, RUN_DIR)
print('僅本地下載，未發布:', zip_path)
files.download(zip_path)


## 方法來源與實際驗證範圍

- [LoRA / Microsoft，2021](https://github.com/microsoft/LoRA)：少量 adapter；500M/T4 預設足夠簡單。
- [QLoRA / University of Washington NLP，2023](https://github.com/artidoro/qlora)：NF4、double quantization；較省 GPU 記憶體。論文模型/任務結果不能直接當本模型的學測表現。
- [PiSSA，2024](https://github.com/GraphPKU/PiSSA)：主奇異向量初始化；[作者張牧涵的個人頁](https://muhanzhang.github.io/)說明其北京大學教師職務。
- [LoftQ，2023 / ICLR 2024](https://github.com/yxli2123/LoftQ)：初始化以降低量化誤差；本格選項是[PEFT one-step replacement](https://huggingface.co/docs/peft/developer_guides/quantization)，不同於完整迭代版。
- [PEFT 近期方法與限制](https://github.com/huggingface/peft/blob/main/docs/source/package_reference/lora.md)：2025 aLoRA 無法合併；2026 Astra/MiCA 需要新版 API 或校準，MiCA 偏向 base 模型續訓，未偷偷加入此固定版本流程。
- [Optimum ONNX](https://github.com/huggingface/optimum-onnx/tree/v0.1.0)、[Transformers.js 3.8.1](https://github.com/huggingface/transformers.js/tree/3.8.1)、[ORT MatMulNBits](https://github.com/microsoft/onnxruntime/blob/v1.21.0/onnxruntime/python/tools/quantization/matmul_4bits_quantizer.py)：部署格式與算子來源。

儲存庫的純 Python 測試驗證資料 schema、分割、prompt 相同、mask、重複比對與發布門檻；不代表雲端已完成 GPU 微調或整模型 ONNX 測試。本筆記本的實際結果由你的 Colab 執行留下。iPhone 13 真機測試需另外進行。
